# Causal Inference Crash Course:
## Part 6 — Panel Models: DiD, Synthetic Control & Synthetic DiD
Julian Hsu

## Causal Inference Series
1. Foundations
2. Causal Models
3. Inference
4. Best Practices
5. Heterogeneous Treatment Effects
6. **Panel Models  ← this deck**
7. Regression Discontinuity
8. Arguable Validation

## Overview
* Panel methods use **repeated observations of the same units over time** to
  estimate causal effects — the workhorse of quasi-experimental design.
    * A quarter of NBER working papers and 16% of top-five journal articles use
      difference-in-differences (Currie, Kleven, Zwiers 2020).
* We cover three ways to build the **counterfactual** for a treated unit:
    * **Difference-in-Differences (DiD)** — a parallel control trend
    * **Synthetic Control (SC)** — a weighted average of controls
    * **Synthetic DiD (SDID)** — a hedge that borrows from both
* Throughout: define the **estimand first**, state the assumptions plainly,
  then estimate — and stress-test everything in a simulated worked example.
* Assumes the earlier decks: potential outcomes, OLS, inference.

# Panel Data

## The big picture
![Image](Figures/Panel_Figure_0.png)
* We track a **treated** unit and many **control** units over time, and see
  their outcomes before and after treatment.
* If, absent treatment, the treated unit's trend would have matched the
  controls, then the **gap that opens after treatment is the effect**.

## The target parameter: the ATT
* Each unit has two potential outcomes at time $t$: $Y_{it}(1)$ if treated,
  $Y_{it}(0)$ if not. We only ever see one.
* The problem is the **missing $Y_{it}(0)$** for treated units after treatment.
* Panel methods target the **Average Treatment effect on the Treated**:
$$ \tau_{ATT} = E\big[\,Y_{it}(1) - Y_{it}(0)\ \big|\ W_i = 1,\ t \ge 0\,\big] $$
* This is **not** the ATE. We learn the effect *for the units that were
  treated*, by reconstructing their counterfactual $Y_{it}(0)$.
* Every model below is just a different way to **predict that counterfactual**.

## Comparing trends lets us *arguably* validate
* We want to know how the treated unit would behave **if we had not treated it**.
* We can never check this directly — but we *can* check the **pre-treatment**
  period, where the counterfactual is observed.
* The more the control trend matches the treated trend **before** treatment,
  the more we trust it to stand in for $Y_{it}(0)$ **after**.
* This is the recurring move in every panel method: earn trust in the
  pre-period, then extrapolate.

## Where DiD, SC, and SDID come in
* How do we build that ideal control trend — and what if no single control
  unit looks like the treated one?

| Method | Counterfactual is… | Trusts… |
|:--|:--|:--|
| **DiD** | the control group shifted to the treated unit's level | parallel trends |
| **Synthetic Control** | a **weighted** average of controls matching the pre-period | pre-period fit |
| **Synthetic DiD** | a weighted control **plus** a DiD-style level shift | a bit of both |

* We build them in that order, because each one relaxes a weakness of the last.

# Difference-in-Differences

## DiD: we predict the counterfactual
* The treated unit's missing $Y_{it}(0)$ is predicted by taking its **own
  pre-treatment level** and adding the **change the control group experienced**.
* In words: *"the treated units would have moved just like the controls did."*
![Image](Figures/Panel_Figure_1.png)

## Setup and notation
* Let's set up notation:
    * $W_i \in \{0,1\}$ — whether unit $i$ is **ever** treated.
    * $\text{post}_t = 1\{t \ge 0\}$ — whether period $t$ is after treatment.
    * $D_{it} = W_i \cdot \text{post}_t$ — the **treatment indicator** (on only
      for treated units, in post periods).
* The data-generating model we have in mind:
$$ Y_{it} = \tau D_{it} + \gamma_t + \eta_i + \zeta_{it} $$
    * $\eta_i$: a **unit** fixed effect (time-invariant level of unit $i$)
    * $\gamma_t$: a **time** fixed effect (common shock in period $t$)
    * $\zeta_{it}$: everything else — the idiosyncratic error

## Two naïve comparisons, and why each fails
* **Post vs. Pre** (treated units only): compare $Y_{it}$ after vs. before.
    * Fails: $\gamma_t$ moved too — you can't tell the treatment from the trend.
* **Treated vs. Control** (post only): compare treated to control after treatment.
    * Fails: $\eta_i$ differs — treated and control units start at different levels.
* Each naïve comparison confounds the treatment with *one* nuisance term.
  The trick is to **difference both away**.

## DiD comes from combining two differences
$$ Y_{it} = \tau D_{it} + \gamma_t + \eta_i + \zeta_{it} $$
* **Post − Pre**, separately within the treated and the control groups:
$$ \Delta_T = E[Y_{it}\mid t\ge0, W_i{=}1] - E[Y_{it}\mid t<0, W_i{=}1] $$
$$ \Delta_C = E[Y_{it}\mid t\ge0, W_i{=}0] - E[Y_{it}\mid t<0, W_i{=}0] $$
* Differencing over time **cancels the unit effect** $\eta_i$, leaving only the
  time term and the error:
$$ \Delta_T = \tau + (\gamma_{t\ge0}-\gamma_{t<0}) + \text{errors}, \qquad
   \Delta_C = (\gamma_{t\ge0}-\gamma_{t<0}) + \text{errors} $$

## Deriving DiD: the second difference
$$ \Delta_T = \tau + (\gamma_{t\ge0}-\gamma_{t<0}) + \bar\zeta_T, \qquad
   \Delta_C = (\gamma_{t\ge0}-\gamma_{t<0}) + \bar\zeta_C $$
* The **time term $\gamma_t$ is common** to both groups — so subtract them:
$$ \Delta_T - \Delta_C = \tau + (\bar\zeta_T - \bar\zeta_C) $$
* If treated and control units do **not** differ in time-varying ways
  ($E[\bar\zeta_T - \bar\zeta_C]=0$), the second difference **is** the ATT:
$$ \boxed{\ \Delta_T - \Delta_C = \tau_{ATT}\ } $$

## Estimating DiD: two-way fixed effects
* A simple version aggregates the fixed effects into group/period dummies:
$$ Y_{it} = \tau D_{it} + \gamma\,\text{post}_t + \eta\,W_i + \epsilon_{it} $$
* The current standard is the **two-way fixed effects (TWFE)** regression, with
  a full set of unit and time fixed effects:
$$ Y_{it} = \tau D_{it} + \gamma_t + \eta_i + \epsilon_{it} $$
* The coefficient $\hat\tau$ on $D_{it}$ **is** the diff-in-diff estimate — same
  number as the two differences, with more precision and easy covariates.
* In `panelib`: `did.twfe(...)`.

## The assumptions, stated plainly
* DiD identifies the ATT under **two** assumptions — keep them separate:
* **1. Parallel trends.** Absent treatment, the treated group's average outcome
  would have moved *parallel* to the control group's:
$$ E[Y_{it}(0)\mid W_i{=}1] - E[Y_{it}(0)\mid W_i{=}0]\ \text{is constant in } t. $$
* **2. No anticipation.** Units don't change behavior *before* treatment starts
  (no effect leaking into the pre-period), so the pre-period identifies $Y(0)$.
* A subtlety: parallel trends is **functional-form dependent** — it can hold in
  levels but not in logs. Pick the scale your estimand is about (Roth &
  Sant'Anna 2023).

## DiD estimates the ATT, not the ATE
* The second difference recovers $\tau$ **only for the treated units** — their
  realized trend is what we differenced.
* Nothing here tells us the effect on a control unit that was never treated;
  their potential response could differ.
* This is a *feature*, not a bug: the ATT is usually the policy-relevant number
  ("what did the rollout do to the units we rolled it out to?").

## The event study: check trends period-by-period
* Instead of one pre and one post, estimate a **separate coefficient for each
  period** relative to treatment ($k=0$):
$$ Y_{it} = \sum_{k\ne -1} \tau_k\, 1\{t - t^*_i = k\} + \gamma_t + \eta_i + \epsilon_{it} $$
![Image](Figures/Panel_Figure_3.png)
* **Pre-period $\tau_k$ ($k<0$) should sit at zero** — that is the parallel-trends
  check. **Post-period $\tau_k$** traces the dynamic effect.

## Validating DiD: is the parallel-trends assumption plausible?
* **1. Eyeball it.** Plot treated vs. control; do the pre-treatment trends move
  together? (Figure below, left = yes, right = no.)
* **2. Placebo pre-trend test.** In the event study, jointly test that the
  pre-period coefficients are zero.
![Image](Figures/Panel_Figure_2.png)

## A caution on pre-trend tests
* A passing pre-trend test is **reassuring, not proof** — the assumption is
  about the *unobserved* post-period counterfactual, which no test can see.
* **Pre-tests have low power** (Roth 2022): the trend violations most likely to
  bias you are exactly the ones a pre-test is least likely to reject. And
  conditioning on "passing" can itself distort your estimate.
* Best practice: report the event study, and probe **sensitivity** to plausible
  violations rather than treating a non-rejection as a green light
  (Rambachan & Roth 2023).

## Including time-varying covariates
* Parallel trends may hold only **after conditioning** on covariates $X_{it}$
  (e.g. seasonality, unit size).
* Add them to the TWFE regression — but a plain linear control can be biased if
  effects are heterogeneous.
* Modern practice uses **doubly-robust DiD** (Sant'Anna & Zhao 2020): combine an
  outcome-regression counterfactual with a propensity model, so you only need
  *one* of them right.

## ⚠️ Staggered adoption breaks plain TWFE
* When units are treated **at different times**, the single TWFE coefficient
  becomes a weighted average of all possible 2×2 comparisons — including ones
  that use **already-treated units as controls** ("forbidden comparisons").
* With dynamic effects, those weights can even go **negative**, so $\hat\tau$ can
  have the wrong sign (Goodman-Bacon 2021; de Chaisemartin & D'Haultfœuille 2020).
* Fix: **heterogeneity-robust estimators** that only compare treated to
  not-yet-treated units (Callaway & Sant'Anna 2021). See the
  [**appendix**](#/appendix-staggered-adoption-and-forbidden-comparisons).

# Synthetic Control

## SC: we predict the counterfactual — as a weighted average
* DiD shifts the *whole* control group to the treated unit's level. But what if
  **no single control** looks like the treated unit?
* **Synthetic control** builds a bespoke control: a **weighted average of donor
  units** that tracks the treated unit's pre-treatment path.
![Image](Figures/Panel_Figure_4.png)

## Setup and notation
* Let unit $i=0$ be the **treated** unit; units $i=1,\dots,J$ are the **donor
  pool** of untreated controls.
* We predict the treated counterfactual as an intercept plus a weighted sum of
  donors:
$$ \hat Y_{0t}(0) = \mu + \sum_{j=1}^{J} \omega_j\, Y_{jt} $$
* Choose $(\mu, \omega)$ so the synthetic unit **matches the treated unit's
  pre-treatment outcomes** $Y_{0t}$ for $t<0$.
* The estimated effect is the post-period gap: $\hat\tau_t = Y_{0t} - \hat Y_{0t}(0)$.

## The big idea: a data-driven control group
* Rather than assume a parallel trend, SC **lets the data pick** which controls,
  and in what proportions, reproduce the treated unit's history.
* Good pre-period fit is **checkable** — you can see whether the synthetic unit
  tracks the treated one before treatment.
* The weights are usually **sparse**: a handful of donors do the work.
![Image](Figures/Panel_Figure_5.png)

## Abadie–Diamond–Hainmueller (2010): the classic SC
* The original SC restricts the weights to a **simplex**: non-negative and
  summing to one, with no intercept.
$$ \min_{\omega}\ \sum_{t<0}\Big(Y_{0t} - \textstyle\sum_j \omega_j Y_{jt}\Big)^2
   \quad\text{s.t.}\quad \omega_j \ge 0,\ \sum_j \omega_j = 1 $$
* This keeps the counterfactual **inside the convex hull** of the donors — no
  extrapolation, interpretable weights.
* Best when the treated unit is *bracketed* by the donor pool. In `panelib`:
  `sc.sc_model(model_name='adh', ...)`.

## Doudchenko–Imbens (2016): a less restrictive SC
* Relax the ADH constraints: allow an **intercept $\mu$** (a level shift, like
  DiD) and let the weights be estimated with an **elastic-net penalty** instead
  of the simplex.
$$ \min_{\mu,\omega}\ \sum_{t<0}\Big(Y_{0t}-\mu-\textstyle\sum_j\omega_j Y_{jt}\Big)^2
   + \lambda\Big[(1-\alpha)\|\omega\|_2^2 + \alpha\|\omega\|_1\Big] $$
* The penalty $(\lambda,\alpha)$ is chosen by **cross-validation** on held-out
  pre-periods, trading off fit against over-fitting the donor weights.
* More flexible, but can extrapolate. In `panelib`: `sc.sc_model(model_name='di', ...)`.

## How does inference work for SC?
* With one treated unit there is no cross-sectional sample to form a standard
  error the usual way. Two common answers:
* **Placebo / permutation** (ADH): re-run SC pretending **each donor** was the
  treated unit. If the real treated gap is extreme relative to this placebo
  distribution, it's significant (a Fisher-style exact test).
* **Conformal inference** (Chernozhukov, Wüthrich, Zhu 2021): invert a test that
  permutes residuals across time blocks — valid with a single treated unit, and
  the procedure `panelib` uses for **all** the models here, so the comparison is
  apples-to-apples.

## Validating SC and adding covariates
* **Pre-period fit** is the first diagnostic: a synthetic unit that can't track
  the pre-period won't be trusted post-period. `panelib` reports pre-train and
  pre-test fit (MAPE) so you can catch over-fitting.
* **Backdating / hold-out**: fit on early pre-periods, check the fit on the held-
  out late pre-periods.
* **Covariates**: ADH can match on pre-treatment covariates as well as outcomes;
  DI-style models fold them into the donor design matrix. See Abadie (2021) for
  a full review.

# Synthetic Difference-in-Differences

## Motivation: why hedge?
* **DiD** trusts *parallel trends* — one bad assumption and it can break badly.
* **Synthetic control** trusts *pre-period fit* — great when a good donor mix
  exists, but it can extrapolate and it ignores the level information DiD uses.
* **Synthetic DiD** (Arkhangelsky, Athey, Hirshberg, Imbens, Wager 2021) is a
  **hedge**: keep DiD's double-differencing, but *weight* the controls (like SC)
  and *weight* the time periods, so parallel trends only has to hold for the
  weighted comparison.

## SDID: we predict the counterfactual — weighted, then differenced
* SDID runs a **weighted two-way fixed effects** regression:
$$ \hat\tau = \arg\min_{\tau,\mu,\eta,\gamma}\ \sum_{i,t}
   \big(Y_{it}-\mu-\eta_i-\gamma_t-\tau D_{it}\big)^2\, \hat\omega_i\, \hat\lambda_t $$
* Two sets of weights do the hedging:
    * **Unit weights $\hat\omega_i$** — SC-style, pick controls whose *trend*
      matches the treated unit (an intercept is allowed, so only trends matter).
    * **Time weights $\hat\lambda_t$** — pick pre-periods that look like the
      post-period, down-weighting irrelevant history.
![Image](Figures/Panel_Figure_6.png)

## What the two weightings buy you
* **Unit weights** make parallel trends only need to hold **between the treated
  unit and its synthetic match**, not the whole donor pool — a much weaker ask.
* **Time weights** focus the "before" comparison on the pre-periods most like
  the post-period, reducing sensitivity to distant, irrelevant history.
* The DiD double-difference is still there, so SDID keeps DiD's robustness to
  *level* differences that pure SC has to fit away.
* In `panelib`: `sdid.twfe_sdid(...)`, with `sdid.conformal_inference(...)` for
  the CI — the same conformal procedure as the other models.

## When does SDID help?
* SDID **shines when parallel trends is shaky but a good weighted control
  exists** — it often has the lowest bias *and* keeps close-to-nominal coverage.
* It **reduces to DiD** if the data want uniform weights, and behaves **like SC**
  if the level shift is unnecessary — so it rarely does much *worse* than either.
* Cost: it needs a reasonable number of pre-periods to estimate the time
  weights, and inference is a bit more involved. We will see all of this in the
  worked example.

## DiD vs. SC vs. SDID
| | DiD | Synthetic Control | Synthetic DiD |
|:--|:--|:--|:--|
| Control group | unweighted average of all controls, allowing a level shift | weighted subset matching the treated pre-period | weighted controls **and** a level shift |
| Key assumption | parallel trends (all controls) | good pre-period fit | parallel trends for the **weighted** match |
| Data it wants | many units *or* many periods | long pre-period, good donors | both, ideally |
| Weights | none (uniform) | unit weights $\omega$ | unit **and** time weights $\omega,\lambda$ |
| Inference | OLS / clustered; here, conformal | permutation / conformal | conformal |
| Main failure mode | non-parallel trends | no donor mix fits; extrapolation | too few pre-periods |

# Worked Example

## Simulated data, so we know the answer
* We simulate a panel with **one treated unit** and **100 controls**, 20
  pre-periods and a short post-period, using `panelib`'s
  `dgp.simulate_panel`. Because we built it, we know the **true ATT**.
* A shared **AR(1) common trend** ($\rho=0.8$) drives all units, plus unit
  effects and noise. Two scenarios:
    1. **Parallel trends hold** ($\sigma_\lambda = 0$): no unit-specific trends.
    2. **Parallel trends violated** ($\sigma_\lambda = 1$): each unit gets its
       own linear trend, so the treated unit drifts away from the controls.
* We fit **DiD, SC-ADH, SC-DI, and SDID**, all with the *same* conformal
  inference. Companion notebook with all code
  [here](https://github.com/shoepaladin/causalinference_crashcourse/blob/main/Notebooks/6%20Panel%20Models.ipynb).

## Scenario 1: parallel trends hold
* Observed treated outcome (solid) vs. each model's predicted counterfactual
  (dashed), with a 95% conformal band in the post-period.
![Image](Figures/Panel_Figure_7.png)
* All three track the treated unit's pre-period and land near the true ATT —
  when parallel trends holds, every method works.

## Scenario 2: parallel trends violated
* Now the treated unit has its own downward trend the controls don't share.
![Image](Figures/Panel_Figure_8.png)
* **DiD's counterfactual ignores that trend and misses badly.** SC-ADH and SDID
  weight the donors to reproduce the treated unit's trajectory, so their
  counterfactuals bend with it and stay close to the truth.

## Bias and coverage across ~200 simulations per scenario
![Image](Figures/Panel_Figure_9.png)
* **Left — bias.** Under parallel trends every model is roughly unbiased. Under a
  violation, **DiD is badly biased** while SC and SDID stay close to zero.
* **Right — coverage.** DiD's 95% interval covers the truth **96%** of the time
  when trends are parallel, but collapses to **2%** when they aren't. SC and SDID
  keep coverage far closer to nominal — **SDID highest under the violation (85%)**.

## Reading the results
* **DiD** is efficient and unbiased *when its one assumption holds* — and
  catastrophic when it doesn't. Its confidence interval doesn't warn you: it
  stays tight while pointing at the wrong answer.
* **Synthetic control** is robust to the trend violation because it fits the
  treated trajectory directly — at the cost of wider, model-dependent inference.
* **SDID** gets the best of both here: low bias under the violation *and* the
  coverage closest to nominal, because the weighting makes its parallel-trends
  requirement much weaker.

## Conclusion — which method, when?
* Start from the **counterfactual** you can defend, not the estimator you like.
* **DiD**: reach for it when parallel trends is plausible and you have clean
  timing. Cheap, transparent, and the event study makes the assumption visible.
* **Synthetic control**: when one treated unit (or a few) has a long pre-period
  and a good donor pool — and you can *show* the pre-period fit.
* **Synthetic DiD**: when you're worried about trends but have the panel to
  support weighting — often the safest default.
* Whatever you pick: **state the assumption, test what you can, and stress-test
  what you can't.**

# Appendix

## Appendix: panel data's advantage over cross-sections
* With a single cross-section, a level difference between treated and control
  units ($\eta_i$) is indistinguishable from a treatment effect.
* Panel data lets us **difference out** the time-invariant $\eta_i$ by comparing
  each unit to its own past — the core reason panel methods can beat
  selection-on-observables.
* The price is a **new** assumption about *time-varying* differences (parallel
  trends), which is what the rest of this deck is about.

## Appendix: staggered adoption and forbidden comparisons
* With a single treatment date, TWFE = the clean 2×2 DiD. With **staggered**
  timing, Goodman-Bacon (2021) shows $\hat\tau^{TWFE}$ is a variance-weighted
  average of **all** 2×2 comparisons between timing groups.
* Some of those comparisons use **already-treated units as the control group**.
  If treatment effects grow over time, the already-treated unit's *rising* effect
  enters with a **negative weight**, biasing $\hat\tau$ — possibly to the wrong sign.
* **Heterogeneity-robust estimators** (Callaway & Sant'Anna 2021; Sun & Abraham
  2021; de Chaisemartin & D'Haultfœuille 2020) fix this by only ever comparing
  newly-treated units to **not-yet-treated** ones, then aggregating clean
  group-time effects $ATT(g,t)$.

## Appendix: how the conformal inference works
* All models here share one inference procedure, so their intervals are
  comparable (`conformal_inf` in `panelib`).
* **Idea**: under the null $\tau_t = \theta$, the post-period residuals (observed
  − counterfactual − $\theta$) should look exchangeable with the pre-period
  residuals. Permute residuals across time blocks and compute a test statistic.
* Invert the test over a grid of $\theta$ to get the confidence interval; the
  reported SE is the CI half-width $/\,(2\times1.96)$.
* Because it keys off **pre-period fit**, models that fit the pre-period poorly
  (DiD under a trend violation) get **wider, honest** intervals — usually.

## Appendix: estimating the SDID weights
* **Unit weights $\hat\omega$** solve an SC-style problem *with an intercept and a
  small ridge penalty*, matching control pre-period trends to the treated unit:
$$ \min_{\omega_0,\omega\ge0,\,\mathbf 1'\omega=1}\ \sum_{t<0}\Big(\omega_0 +
   \textstyle\sum_j \omega_j Y_{jt} - Y_{0t}\Big)^2 + \zeta^2 T_{pre}\|\omega\|_2^2 $$
* **Time weights $\hat\lambda$** solve the analogous problem *across time*, finding
  the pre-periods whose control outcomes best predict the post-period.
* Plug both into the **weighted TWFE** regression; the coefficient on $D_{it}$ is
  the SDID ATT. `panelib`: `sdid.twfe_sdid` returns `omega_weights`,
  `lambda_weights`, and the counterfactual.

## Literature Review of Related Papers
* **DiD — foundations & modern advances**
    * Roth, Sant'Anna, Bilinski, Poe (2023). "What's Trending in DiD? A
      Synthesis of the Recent Econometrics Literature." *J. Econometrics.*
      [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407623001318)
    * Goodman-Bacon (2021). "DiD with Variation in Treatment Timing."
      *J. Econometrics.* [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407621001445)
    * Callaway & Sant'Anna (2021). "DiD with Multiple Time Periods."
      *J. Econometrics.* [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407620303948)
    * de Chaisemartin & D'Haultfœuille (2020). "Two-Way FE Estimators with
      Heterogeneous Treatment Effects." *AER.* [link](https://www.aeaweb.org/articles?id=10.1257/aer.20181169)
    * Sun & Abraham (2021). "Estimating Dynamic Treatment Effects in Event
      Studies." *J. Econometrics.* [link](https://www.sciencedirect.com/science/article/abs/pii/S030440762030378X)
* **DiD — inference, covariates, and pre-tests**
    * Sant'Anna & Zhao (2020). "Doubly Robust DiD Estimators."
      *J. Econometrics.* [link](https://www.sciencedirect.com/science/article/abs/pii/S0304407620301901)
    * Roth (2022). "Pre-test with Caution: Event-Study Estimates After Testing
      for Parallel Trends." *AER: Insights.* [link](https://www.aeaweb.org/articles?id=10.1257/aeri.20210236)
    * Rambachan & Roth (2023). "A More Credible Approach to Parallel Trends."
      *REStud.* [link](https://academic.oup.com/restud/article/90/5/2555/7039335)

## Literature Review (continued)
* **Synthetic control**
    * Abadie, Diamond, Hainmueller (2010). "Synthetic Control Methods for
      Comparative Case Studies." *JASA.* [link](https://www.tandfonline.com/doi/abs/10.1198/jasa.2009.ap08746)
    * Doudchenko & Imbens (2016). "Balancing, Regression, DiD and Synthetic
      Control Methods: A Synthesis." *NBER w22791.* [link](https://www.nber.org/papers/w22791)
    * Abadie (2021). "Using Synthetic Controls: Feasibility, Data Requirements,
      and Methodological Aspects." *J. Economic Literature.* [link](https://www.aeaweb.org/articles?id=10.1257/jel.20191450)
    * Ben-Michael, Feller, Rothstein (2021). "The Augmented Synthetic Control
      Method." *JASA.* [link](https://www.tandfonline.com/doi/full/10.1080/01621459.2021.1929245)
* **Synthetic DiD & shared inference**
    * Arkhangelsky, Athey, Hirshberg, Imbens, Wager (2021). "Synthetic
      Difference-in-Differences." *AER.* [link](https://www.aeaweb.org/articles?id=10.1257/aer.20190159)
    * Chernozhukov, Wüthrich, Zhu (2021). "An Exact and Robust Conformal Inference
      Method for Counterfactual and Synthetic Controls." *JASA.* [link](https://www.tandfonline.com/doi/full/10.1080/01621459.2021.1920957)
* **Companion code:** `panelib.py` in
  [statanomics](https://github.com/shoepaladin/statanomics/blob/main/workingcode/panelmodels/panelib.py).

In [ ]:
# --- Figure generation (run this notebook to regenerate Figures/Panel_Figure_*.png) ---
# Schematic figures (0-6) illustrate the ideas; worked-example figures (7-9) are
# produced from panelib fits on simulated data, so they are fully reproducible.
# panelib is the canonical panel library in the *statanomics* repo; we import it
# from a sibling clone (and shallow-clone it if it isn't already present) so the
# code here is never duplicated.
import os, sys, io, subprocess, warnings
from contextlib import redirect_stdout
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch


def _ensure_panelib():
    here = os.getcwd()
    cands = [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here)),
             os.path.expanduser("~")]
    for base in cands:
        p = os.path.join(base, "statanomics", "workingcode", "panelmodels")
        if os.path.exists(os.path.join(p, "panelib.py")):
            return p
    dest = os.path.join(os.path.dirname(here), "statanomics")
    if not os.path.exists(dest):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/shoepaladin/statanomics.git", dest],
                       check=True)
    return os.path.join(dest, "workingcode", "panelmodels")

sys.path.insert(0, _ensure_panelib())
from panelib import did, sc, sdid, dgp

# Palette — matches the deck theme (theme/custom.css) and the RD deck figures.
ACCENT, ACCENT2, CTRLC = "#0e7490", "#0f766e", "#94a3b8"
INK, MUTED, LINE = "#17222e", "#5b6b7a", "#cbd5e1"
RED, AMBER, VIOLET = "#b91c1c", "#d97706", "#6d28d9"
OBS = "#334155"
EST = {"DiD": RED, "SC-ADH": ACCENT, "SC-DI": AMBER, "SDID": VIOLET}

FIGDIR = os.path.join(os.getcwd(), "Figures")
os.makedirs(FIGDIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 11,
                     "axes.grid": True, "grid.alpha": .25,
                     "axes.edgecolor": "#cbd5e1", "axes.linewidth": .8})
DD = {"treatment": "treated", "date": "time", "post": "post",
      "unitid": "unit_id", "outcome": "y"}


def _save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, name), bbox_inches="tight")
    plt.close(fig)


def _mark_treat(ax, t, label="treatment"):
    ax.axvline(t, color=MUTED, ls="--", lw=1.2)
    ax.annotate(label, xy=(t, 0.97), xycoords=("data", "axes fraction"),
                xytext=(4, 0), textcoords="offset points", color=MUTED,
                fontsize=9, ha="left", va="top")

In [ ]:
# ---------------------------------------------------------------- Figure 0
# Big picture: we track a treated unit and control units over time.
def fig0():
    rng = np.random.default_rng(6)
    T, t0 = 24, 16
    t = np.arange(T)
    trend = 8 + 0.18 * t + 0.6 * np.sin(t / 3)
    ctrl = trend + rng.normal(0, .25, (6, T)) + rng.uniform(-1.2, 1.2, (6, 1))
    treated = trend + 0.4 + rng.normal(0, .2, T)
    treated[t0:] += 2.4                                   # the effect turns on
    fig, ax = plt.subplots(figsize=(8.4, 3.7))
    for c in ctrl:
        ax.plot(t, c, color=CTRLC, lw=1.1, alpha=.7)
    ax.plot([], [], color=CTRLC, lw=1.4, label="control units")
    ax.plot(t, treated, color=ACCENT, lw=2.4, marker="o", ms=3.5,
            label="treated unit")
    _mark_treat(ax, t0)
    ax.set_xlabel("time"); ax.set_ylabel("outcome  Y (e.g. purchases)")
    ax.set_title("We track a treated unit and many controls over time")
    ax.legend(loc="upper left", framealpha=.9)
    _save(fig, "Panel_Figure_0.png")


# ---------------------------------------------------------------- Figure 1
# The two differences: DiD as a 2x2 with a parallel-trends counterfactual.
def fig1():
    pre, pst = 0, 1
    c_pre, c_pst = 6.0, 7.3                               # control moves +1.3
    t_pre = 8.0                                           # treated starts higher
    cf_pst = t_pre + (c_pst - c_pre)                      # parallel counterfactual
    t_pst = cf_pst + 2.5                                  # true jump = ATT
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    ax.plot([pre, pst], [c_pre, c_pst], "-o", color=CTRLC, lw=2.2, ms=7,
            label="control")
    ax.plot([pre, pst], [t_pre, t_pst], "-o", color=ACCENT, lw=2.4, ms=7,
            label="treated (observed)")
    ax.plot([pre, pst], [t_pre, cf_pst], "--o", color=ACCENT2, lw=2.0, ms=6,
            mfc="white", label="treated counterfactual\n(parallel to control)")
    ax.add_patch(FancyArrowPatch((pst, cf_pst), (pst, t_pst),
                 arrowstyle="<->", mutation_scale=13, color=RED, lw=1.8))
    ax.annotate("ATT", xy=(pst, (cf_pst + t_pst) / 2), xytext=(10, 0),
                textcoords="offset points", color=RED, fontsize=13,
                fontweight="bold", va="center")
    ax.set_xticks([pre, pst]); ax.set_xticklabels(["pre  (t<0)", "post  (t≥0)"])
    ax.set_ylabel("outcome  Y"); ax.set_xlim(-.25, 1.5); ax.grid(axis="x")
    ax.set_title("Diff-in-diff: the gap vs. the parallel counterfactual")
    ax.legend(loc="upper left", framealpha=.9, fontsize=9.5)
    _save(fig, "Panel_Figure_1.png")


# ---------------------------------------------------------------- Figure 2
# Parallel vs non-parallel pre-trends.
def fig2():
    rng = np.random.default_rng(3)
    T, t0 = 16, 10
    t = np.arange(T)
    fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.7), sharey=True)
    for ax, mode in zip(axes, ["parallel", "nonparallel"]):
        base = 6 + 0.2 * t
        ctrl = base + rng.normal(0, .12, T)
        if mode == "parallel":
            treated = base + 1.6 + rng.normal(0, .12, T)
        else:
            treated = 6 + 0.5 * t + rng.normal(0, .12, T)   # steeper pre-trend
        treated[t0:] += 2.2
        ax.plot(t, ctrl, "-o", color=CTRLC, lw=2, ms=3.5, label="control")
        ax.plot(t, treated, "-o", color=ACCENT, lw=2.2, ms=3.5, label="treated")
        _mark_treat(ax, t0)
        ax.set_xlabel("time")
        ax.set_title("Parallel trends: YES" if mode == "parallel"
                     else "Parallel trends: NO")
    axes[0].set_ylabel("outcome  Y")
    axes[0].legend(loc="upper left", framealpha=.9, fontsize=9.5)
    _save(fig, "Panel_Figure_2.png")


# ---------------------------------------------------------------- Figure 3
# Event study: real panelib event_study coefficients. Clean vs pre-trend.
def _event_df(sigma_lambda, seed, T_post=8):
    df, att = dgp.simulate_panel(seed=seed, T_pre=12, T_post=T_post,
                                 N_control=60, noise_sd=0.3, att_pct=0.6,
                                 rho=0.8, sigma_lambda=sigma_lambda)
    with redirect_stdout(io.StringIO()):
        r = did.twfe(data=df, data_dict=DD)
    es = r["event_study"].copy()
    es["time_period"] = pd.to_numeric(es["time_period"])
    es["pre_event"] = pd.to_numeric(es["pre_event"])
    es["k"] = es["time_period"] - 12          # event time (treatment at k=0)
    return es, att


def fig3():
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), sharey=True)
    for ax, (sig, seed, ttl) in zip(
            axes, [(0.0, 5, "Clean: flat pre-trend"),
                   (0.9, 5, "Pre-trend violation")]):
        es, att = _event_df(sig, seed)
        k = es["k"].to_numpy(); b = es["coef_"].to_numpy()
        se = es["se_"].to_numpy()
        pre = es["pre_event"].to_numpy() == 1
        ax.axhline(0, color=MUTED, lw=1)
        ax.axvline(-0.5, color=MUTED, ls="--", lw=1.2)
        ax.errorbar(k[pre], b[pre], yerr=1.96 * se[pre], fmt="o", color=MUTED,
                    ms=4, lw=1, capsize=2, label="pre (placebo)")
        ax.errorbar(k[~pre], b[~pre], yerr=1.96 * se[~pre], fmt="o",
                    color=ACCENT, ms=5, lw=1.4, capsize=2, label="post (effect)")
        ax.set_xlabel("event time  k  (0 = treatment)")
        ax.set_title(ttl)
    axes[0].set_ylabel(r"estimated $\tau_k$")
    axes[0].legend(loc="upper left", framealpha=.9, fontsize=9)
    _save(fig, "Panel_Figure_3.png")


# ---------------------------------------------------------------- Figure 4
# Synthetic control, the big idea: a weighted donor pool tracks the treated.
def fig4():
    df, att = dgp.simulate_panel(seed=11, T_pre=20, T_post=6, N_control=25,
                                 noise_sd=0.2, att_pct=0.9, rho=0.8,
                                 sigma_lambda=0.0)
    with redirect_stdout(io.StringIO()):
        r = sc.sc_model(model_name="adh", data=df, data_dict=DD)
    tid = df.loc[df.treated == 1, "unit_id"].iloc[0]
    pe = r["predict_est"]
    t = pe.index.to_numpy()
    y_obs = pe[tid].to_numpy(); y_sc = pe[f"{tid}_est"].to_numpy()
    t0 = 20
    fig, ax = plt.subplots(figsize=(8.4, 3.9))
    wide = df.pivot_table(index="time", columns="unit_id", values="y")
    for u in wide.columns:
        if u != tid:
            ax.plot(wide.index, wide[u], color=CTRLC, lw=.7, alpha=.35)
    ax.plot([], [], color=CTRLC, lw=1.2, label="donor pool")
    ax.plot(t, y_obs, color=ACCENT, lw=2.4, label="treated (observed)")
    ax.plot(t, y_sc, ls="--", color=ACCENT2, lw=2.2, label="synthetic control")
    ax.axvspan(t0 - .5, t.max(), color=RED, alpha=.06)
    _mark_treat(ax, t0 - .5)
    ax.set_xlabel("time"); ax.set_ylabel("outcome  Y")
    ax.set_title("Synthetic control: a weighted donor pool tracks the treated")
    ax.legend(loc="upper left", framealpha=.9, fontsize=9.5)
    _save(fig, "Panel_Figure_4.png")


# ---------------------------------------------------------------- Figure 5
# Synthetic-control weights are sparse (simplex): a few donors, most ~0.
def fig5():
    df, att = dgp.simulate_panel(seed=11, T_pre=20, T_post=6, N_control=40,
                                 noise_sd=0.3, att_pct=0.5, rho=0.8,
                                 sigma_lambda=0.0)
    r = sdid.twfe_sdid(data=df, data_dict=DD)     # SC-style simplex unit weights
    w = r["omega_weights"].iloc[:, 0].sort_values(ascending=False)
    top = w.head(12)
    fig, ax = plt.subplots(figsize=(8.2, 3.6))
    ax.bar(range(len(top)), top.values, color=ACCENT, width=.72,
           edgecolor="white")
    ax.set_xticks(range(len(top)))
    ax.set_xticklabels(top.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("weight  ωᵢ")
    ax.set_title("A few donors carry the weight; the rest are ~0  "
                 f"({(w > 0.01).sum()} of {len(w)} active)")
    ax.grid(axis="x")
    _save(fig, "Panel_Figure_5.png")


# ---------------------------------------------------------------- Figure 6
# SDID weights: unit weights ω (which donors) and time weights λ (which pre-periods).
def fig6():
    df, att = dgp.simulate_panel(seed=11, T_pre=20, T_post=6, N_control=40,
                                 noise_sd=0.3, att_pct=0.5, rho=0.8,
                                 sigma_lambda=0.0)
    r = sdid.twfe_sdid(data=df, data_dict=DD)
    w = r["omega_weights"].iloc[:, 0].sort_values(ascending=False).head(12)
    lam = r["lambda_weights"].iloc[:, 0]
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.7))
    axes[0].bar(range(len(w)), w.values, color=ACCENT, width=.72,
                edgecolor="white")
    axes[0].set_xticks(range(len(w)))
    axes[0].set_xticklabels(w.index, rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel("weight"); axes[0].grid(axis="x")
    axes[0].set_title("Unit weights  ωᵢ  (which donors)")
    axes[1].bar(range(len(lam)), lam.values, color=VIOLET, width=.85,
                edgecolor="white")
    axes[1].set_xlabel("pre-period"); axes[1].grid(axis="x")
    axes[1].set_title("Time weights  λₜ  (which pre-periods)")
    _save(fig, "Panel_Figure_6.png")


# ------------------------------------------------------- worked-example fits
# Trajectory figures use a clearer illustrative effect (att_pct=0.4) so the
# post-period gap is visible; the *quantitative* comparison lives in the
# Monte-Carlo tables. SC-DI is omitted here — panelib's Doudchenko-Imbens
# elastic net shrinks the donor weights to ~0 on this DGP, giving an
# intercept-only (flat) counterfactual that reads as broken on a slide even
# though its aggregate ATT is unbiased (see the MC tables, where DI is kept).
TRAJ_MODELS = ["DiD", "SC-ADH", "SDID"]


def _fit_all(sigma_lambda, seed=7, grid=None):
    if grid is None:
        grid = np.arange(-9, 9, 0.05)
    df, true_att = dgp.simulate_panel(seed=seed, T_pre=20, T_post=4,
                                      N_control=60, noise_sd=0.2,
                                      att_pct=0.4, rho=0.8,
                                      sigma_lambda=sigma_lambda)
    tid = df.loc[df.treated == 1, "unit_id"].iloc[0]
    t0 = 20
    out = {}
    with redirect_stdout(io.StringIO()):
        # DiD
        rd = did.twfe(data=df, data_dict=DD)
        inf = did.conformal_inference(data=df, data_dict=DD, theta_grid=grid)
        c = rd["twfe_c"]; rows = c[c.unit_id == tid].sort_values("time")
        out["DiD"] = dict(t=rows.time.to_numpy(), obs=rows.y.to_numpy(),
                          cf=rows.y_hat_counterfactual.to_numpy(),
                          att=inf["real_att"], se=inf["se"])
        # SC-ADH
        r = sc.sc_model(model_name="adh", data=df, data_dict=DD,
                        inference={"alpha": 0.05, "theta_grid": grid})
        pe = r["predict_est"]; rr = r["results_df"]
        se = (float(rr.ci_upper.values[0]) - float(rr.ci_lower.values[0])) / (2 * 1.96)
        out["SC-ADH"] = dict(t=pe.index.to_numpy(), obs=pe[tid].to_numpy(),
                             cf=pe[f"{tid}_est"].to_numpy(),
                             att=float(rr.atet.values[0]), se=se)
        # SDID
        rs = sdid.twfe_sdid(data=df, data_dict=DD)
        infs = sdid.conformal_inference(data=df, data_dict=DD, theta_grid=grid)
        cf = rs["counterfactual"]; rr = cf[cf.treated == 1].sort_values("time")
        out["SDID"] = dict(t=rr.time.to_numpy(), obs=rr.y_obs.to_numpy(),
                           cf=rr.y_c.to_numpy(), att=infs["real_att"],
                           se=infs["se"])
    return out, true_att, t0


def _trajectory_grid(sigma_lambda, fname, suptitle):
    out, true_att, t0 = _fit_all(sigma_lambda)
    fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.9), sharex=True)
    for ax, name in zip(axes, TRAJ_MODELS):
        d = out[name]; col = EST[name]
        t, obs, cf, se = d["t"], d["obs"], d["cf"], d["se"]
        post = t >= t0
        ax.plot(t, obs, color=OBS, lw=2.0, label="observed")
        ax.plot(t, cf, ls="--", color=col, lw=2.0, label="counterfactual")
        tp = t[post]
        ax.fill_between(tp, cf[post] - 1.96 * se, cf[post] + 1.96 * se,
                        color=col, alpha=.18, lw=0)
        ax.axvline(t0 - .5, color=MUTED, ls=":", lw=1.1)
        ax.set_title(name, color=col, fontweight="bold")
        ax.set_xlabel("time")
        txt = f"est ATT {d['att']:+.3f}\ntrue ATT {true_att:+.3f}"
        ax.text(.04, .04, txt, transform=ax.transAxes, fontsize=8.5,
                va="bottom", ha="left",
                bbox=dict(boxstyle="round", fc="white", ec=LINE, alpha=.9))
    axes[0].set_ylabel("outcome  Y")
    axes[0].legend(loc="upper left", fontsize=8.5, framealpha=.9)
    fig.suptitle(suptitle, fontsize=12, y=1.02)
    _save(fig, fname)


def fig7():
    _trajectory_grid(0.0, "Panel_Figure_7.png",
                     "Worked example — parallel trends hold (σλ = 0)")


def fig8():
    _trajectory_grid(1.0, "Panel_Figure_8.png",
                     "Worked example — parallel trends violated (σλ = 1)")


# ---------------------------------------------------------------- Figure 9
# Bias (left) and 95% CI coverage (right) by model and scenario, from the MC CSV.
def fig9(csv_path):
    if not os.path.exists(csv_path):
        print("skip fig9: no MC csv yet"); return
    d = pd.read_csv(csv_path)
    order = ["DiD", "SC-ADH", "SC-DI", "SDID"]
    scen = ["parallel", "violated"]
    x = np.arange(len(order)); w = .38
    n_sims = int(d.groupby(["scenario", "model"]).size().min())

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1))

    def grouped(ax, metric, values_of):
        for i, sc_ in enumerate(scen):
            vals = [values_of(sc_, m) for m in order]
            bars = ax.bar(x + (i - .5) * w, vals, w,
                          color=[EST[m] for m in order],
                          alpha=.55 if sc_ == "violated" else 1.0,
                          hatch="//" if sc_ == "violated" else None,
                          edgecolor="white",
                          label="parallel trends" if sc_ == "parallel" else "violated")
            for b, v in zip(bars, vals):
                ax.annotate(f"{v:.2f}", xy=(b.get_x() + b.get_width() / 2,
                            b.get_height()), xytext=(0, 3 if v >= 0 else -11),
                            textcoords="offset points", ha="center",
                            fontsize=7.5, color=INK)
        ax.set_xticks(x); ax.set_xticklabels(order)
        ax.legend(framealpha=.9, fontsize=8.5, loc="best")

    dbias = d.groupby(["scenario", "model"])["bias"].mean()
    dcov = d.groupby(["scenario", "model"])["covered"].mean()
    grouped(axes[0], "bias", lambda s, m: float(dbias.loc[(s, m)]))
    axes[0].axhline(0, color=INK, lw=1)
    axes[0].set_ylabel("mean bias  (est − true ATT)")
    axes[0].set_title("Bias: DiD collapses when trends aren't parallel")

    grouped(axes[1], "cov", lambda s, m: float(dcov.loc[(s, m)]))
    axes[1].axhline(0.95, color=MUTED, ls="--", lw=1.2)
    axes[1].annotate("nominal 95%", xy=(0.62, 0.95),
                     xycoords=("axes fraction", "data"), xytext=(0, 4),
                     textcoords="offset points", fontsize=8, color=MUTED)
    axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("95% CI coverage")
    axes[1].set_title("Coverage: only SC/SDID stay near nominal")

    fig.suptitle(f"Monte-Carlo across {n_sims} simulations per scenario",
                 fontsize=12, y=1.02)
    _save(fig, "Panel_Figure_9.png")

In [ ]:
# Render every figure (schematic first, then the panelib worked example).
fig0(); fig1(); fig2(); fig3(); fig4(); fig5(); fig6()
fig7(); fig8()
fig9(os.path.join(os.getcwd(), "panel_mc_results.csv"))
print("figures written to", FIGDIR)